# MNIST MLP on KD240 — 250 MHz build

Same HLS IP as `../mnist_kd240.bit`, re-implemented with the PL clock
constrained to 250 MHz instead of 100 MHz. Post-route WNS is +0.010 ns, so
250 MHz is *met*, not overclocked into.

Measured on this board:

| PL clock | wrong predictions | throughput |
|---|---|---|
| 250 MHz (signed off) | 0 | 168,698 fps |
| 300 MHz | 0 | 200,333 fps |
| 375 MHz | 0 | **248,122 fps** |
| 500 MHz | 8,977 | past the limit |

Two differences from the v1 notebook: the 7.84 MB host copy into the DMA buffer
happens **once**, outside the timed region (v1 timed it every iteration, hiding
host memcpy inside the FPGA number), and buffers are flushed/invalidated
explicitly rather than by luck.

In [ ]:
from pynq import Overlay, allocate, ps
import numpy as np
import time

N_IMAGES, PIXELS = 10000, 784

overlay = Overlay('mnist_kd240.bit')
regs = overlay.MultilayerPerceptron_0.register_map

# The PL clock persists across sessions -- loading a bitstream does not reset
# it. Set it explicitly instead of assuming the design frequency.
ps.Clocks.fclk0_mhz = 249.997
print('PL  %.3f MHz' % ps.Clocks.fclk0_mhz)
print('CPU %.2f MHz' % ps.Clocks.cpu_mhz)

In [ ]:
in_buf = allocate(shape=(N_IMAGES * PIXELS,), dtype=np.int8)
out_buf = allocate(shape=(N_IMAGES,), dtype=np.int8)
regs.im_1.im = in_buf.device_address
regs.out_r_1.out_r = out_buf.device_address

x_test = (np.load('../x_test.npy') // 32).astype(np.int8).reshape(N_IMAGES, PIXELS)
y_test = np.load('../y_test.npy').astype(np.uint8)
golden = np.frombuffer(open('golden_pred_i8.bin', 'rb').read(), dtype=np.int8)

in_buf[:] = x_test.reshape(-1)   # setup, not inference -- done once
in_buf.flush()
print('loaded', x_test.shape)

In [ ]:
def mnist_hw():
    out_buf[:] = 0
    out_buf.flush()
    regs.CTRL.AP_START = 1
    while regs.CTRL.AP_DONE == 0:
        pass
    out_buf.invalidate()
    return np.array(out_buf)

res = mnist_hw()
print('mismatches vs golden :', int((res != golden).sum()))
print('accuracy             : %.4f' % (res == y_test).mean())

`mismatches vs golden` must be **0**. Accuracy alone will not catch a bad
clock — a design past its timing limit still scores ~0.97 while individual
predictions are wrong.

In [ ]:
t = %timeit -n 1 -r 10 -o mnist_hw()

mhz = ps.Clocks.fclk0_mhz
print('per image  : %.3f us' % (t.average / N_IMAGES * 1e6))
print('throughput : {:,.0f} fps'.format(N_IMAGES / t.average))
print('cycles/img : %.0f at %.0f MHz' % (t.average / N_IMAGES * mhz * 1e6, mhz))

~1482 cycles/image against 1411 from synthesis. The extra is a fixed ~2.7 ms
per batch of cache maintenance and Python polling on `AP_DONE`, not
accelerator time.

In [ ]:
# Clock sweep. The PS IOPLL only emits 1500/N MHz, so these are the only steps.
for target in [249.997, 299.997, 374.996]:
    ps.Clocks.fclk0_mhz = target
    mnist_hw()                                    # warm up, discard
    t0 = time.time()
    r = mnist_hw()
    dt = time.time() - t0
    print('{:7.1f} MHz  wrong={:<6d} {:,.0f} fps'.format(
        ps.Clocks.fclk0_mhz, int((r != golden).sum()), N_IMAGES / dt))

ps.Clocks.fclk0_mhz = 249.997
print('restored to %.3f MHz' % ps.Clocks.fclk0_mhz)

500 MHz is not in the list on purpose: it produces 8,977 wrong predictions out
of 10,000. 375 MHz is the ceiling on this chip at room temperature — bit-exact,
but overclocked, so it carries no guarantee across temperature or silicon lot.
250 MHz is the number Vivado signs off.

## Hardware vs software, on the board

The check above compares against a reference computed on the host. This cell
rebuilds the same quantised network here on the KD240, straight from the Vitis
AI weight dumps, so nothing about the comparison depends on the host tooling.

Accuracy alone is a weak test — two implementations can both score 0.9764 while
disagreeing on which images they get wrong. `hls == py` counts the images where
they agree exactly, and that is the number that has to be 10000.

In [ ]:
import glob

W = [np.loadtxt(f) for f in sorted(glob.glob('../VitisAI/dump_results/dump_results_weights/quant_dense_*_kernel.txt'))]
B = [np.loadtxt(f) for f in sorted(glob.glob('../VitisAI/dump_results/dump_results_weights/quant_dense_*_bias.txt'))]
layers, scales = [784, 128, 256, 10], [512, 256, 256]
W = [W[i].reshape(layers[i], layers[i + 1]) for i in range(3)]

def mnist_sw(images):
    """Same network in numpy, image by image -- the article's formulation."""
    out = []
    for i in range(len(images)):
        d = images[i]
        for j in range(3):
            d = (d @ W[j] + B[j]) // scales[j]
            if j != 2:
                d = d * (d > 0)
        out.append(np.argmax(d))
    return np.array(out)

ps.Clocks.fclk0_mhz = 249.997

t0 = time.time(); res_hls = mnist_hw(); t_hls = time.time() - t0
t0 = time.time(); res_py = mnist_sw(x_test); t_py = time.time() - t0

In [ ]:
print('acc hls  %.4f' % (res_hls == y_test).mean())
print('acc py   %.4f' % (res_py == y_test).mean())
print('hls == py: {}/{} images'.format(int((res_hls == res_py).sum()), N_IMAGES))
print()
print('hls fps  {:>10,.1f}'.format(N_IMAGES / t_hls))
print('py  fps  {:>10,.1f}'.format(N_IMAGES / t_py))
print('speedup  {:>10.1f}x'.format(t_py / t_hls))

`hls == py` must be 10000. If the two accuracies match but this does not, the
hardware is wrong on some images and right on others by luck — exactly what a
clock past its limit looks like.

The speedup uses the article's per-image Python loop as the software baseline.
A vectorised numpy version of the same maths is roughly 40x faster than that
loop, so treat the multiplier as "versus naive Python", not "versus a CPU".